In [0]:

# Check uploaded CSV files

volume_path = "/Volumes/olist/bronze/raw_files/"

display(dbutils.fs.ls(volume_path))


path,name,size,modificationTime
dbfs:/Volumes/olist/bronze/raw_files/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1793614903000
dbfs:/Volumes/olist/bronze/raw_files/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1793614953000
dbfs:/Volumes/olist/bronze/raw_files/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1793614920000
dbfs:/Volumes/olist/bronze/raw_files/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1793614895000
dbfs:/Volumes/olist/bronze/raw_files/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1793614918000
dbfs:/Volumes/olist/bronze/raw_files/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1793614922000
dbfs:/Volumes/olist/bronze/raw_files/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1793614882000
dbfs:/Volumes/olist/bronze/raw_files/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1793614872000
dbfs:/Volumes/olist/bronze/raw_files/product_category_name_translation.csv,product_category_name_translation.csv,2613,1793614871000


In [0]:

from pyspark.sql import functions as F


In [0]:

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(
        "/Volumes/olist/bronze/raw_files/"
        "olist_orders_dataset.csv"
    )
)

display(orders_df.limit(10))


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [0]:

orders_bronze = (
    orders_df
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.lit("olist_orders_dataset.csv")
    )
)

(
    orders_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.bronze.orders")
)

print("Bronze orders table created successfully!")

print("Total records:", orders_bronze.count())


Bronze orders table created successfully!
Total records: 99441


In [0]:

display(
    spark.sql(
        "SELECT * FROM olist.bronze.orders LIMIT 10"
    )
)


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv


In [0]:

files = {
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

for table_name, filename in files.items():

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(volume_path + filename)
    )

    df = (
        df.withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_file",
            F.lit(filename)
        )
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"olist.bronze.{table_name}")
    )

    print(
        f"{table_name}: {df.count()} records loaded"
    )


customers: 99441 records loaded
order_items: 112650 records loaded
order_payments: 103886 records loaded
order_reviews: 104162 records loaded
products: 32951 records loaded
sellers: 3095 records loaded
geolocation: 1000163 records loaded
category_translation: 71 records loaded


In [0]:

display(spark.sql("SHOW TABLES IN olist.bronze"))


database,tableName,isTemporary
bronze,category_translation,false
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,products,false
bronze,sellers,false
